In [1]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import torch.nn.functional as F
import math
from torch.utils.data import Dataset, DataLoader

class PositionalEncoding(nn.Module):
    """Posicional Encoding seno/cosseno do Transformer original"""
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Criar matriz de positional encoding
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        # Termos de frequência
        div_term = torch.exp(torch.arange(0, d_model, 2).float() *
                           (-math.log(10000.0) / d_model))

        # Seno para posições pares, cosseno para ímpares
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


class MultiHeadAttention(nn.Module):
    """Multi-Head Attention implementada corretamente"""
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        # Projeções
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, mask=None):
        batch_size = query.shape[0]

        # 1. Projetar
        Q = self.W_q(query)
        K = self.W_k(key)
        V = self.W_v(value)

        # 2. Reshape para heads
        Q = Q.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(batch_size, -1, self.num_heads, self.head_dim).transpose(1, 2)

        # 3. Calcular atenção
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.head_dim)

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        attn_weights = F.softmax(scores, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # 4. Aplicar aos valores
        output = torch.matmul(attn_weights, V)

        # 5. Juntar heads
        output = output.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)

        # 6. Projeção final
        output = self.out_proj(output)

        return output, attn_weights


class FeedForward(nn.Module):
    """Feed-Forward Network com GELU"""
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
        self.activation = nn.GELU()  # GELU é melhor que ReLU para transformers

    def forward(self, x):
        x = self.fc1(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x


class TransformerBlock(nn.Module):
    """Bloco Transformer completo (Pre-Norm)"""
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()

        # Subcamada 1: Atenção
        self.attention = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)

        # Subcamada 2: FFN
        self.ffn = FeedForward(d_model, d_ff, dropout)
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Pre-Norm (estilo moderno)
        # 1. Atenção com residual
        attn_output, _ = self.attention(self.norm1(x), self.norm1(x), self.norm1(x), mask)
        x = x + self.dropout(attn_output)

        # 2. FFN com residual
        ffn_output = self.ffn(self.norm2(x))
        x = x + self.dropout(ffn_output)

        return x


class TransformerModel(nn.Module):
    """Modelo Transformer completo para geração de texto"""
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers,
                 max_seq_len=512, dropout=0.1):
        super().__init__()

        self.vocab_size = vocab_size
        self.d_model = d_model

        # 1. Embedding de tokens
        self.token_embedding = nn.Embedding(vocab_size, d_model)

        # 2. Positional Encoding (ISSO ESTAVA FALTANDO!)
        self.pos_encoding = PositionalEncoding(d_model, max_seq_len, dropout)

        # 3. Blocos Transformer
        self.transformer_blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

        # 4. Output Head
        self.output_head = nn.Linear(d_model, vocab_size)

        self.dropout = nn.Dropout(dropout)

        # Inicialização dos pesos (importante!)
        self._init_weights()

    def _init_weights(self):
        """Inicialização Xavier/Glorot para melhor convergência"""
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, x, mask=None):
        """
        x: tokens (batch, seq_len)
        mask: máscara causal para geração (batch, seq_len, seq_len)
        """
        # 1. Token embedding + scaling
        x = self.token_embedding(x) * math.sqrt(self.d_model)

        # 2. Adicionar positional encoding
        x = self.pos_encoding(x)

        # 3. Passar pelos blocos Transformer
        for block in self.transformer_blocks:
            x = block(x, mask)

        # 4. Projetar para vocabulário
        logits = self.output_head(x)

        return logits

    def generate_causal_mask(self, size):
        """Criar máscara causal (triangular inferior) para geração"""
        mask = torch.triu(torch.ones(size, size), diagonal=1).bool()
        mask = ~mask  # Invert for 1 = can see, 0 = cannot see
        return mask.unsqueeze(0).unsqueeze(0)  # (1, 1, size, size)

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=50, temperature=1.0, top_k=None):
        """
        Gerar texto token por token
        input_ids: (batch, seq_len) - tokens iniciais
        """
        self.eval()
        batch_size = input_ids.shape[0]

        for _ in range(max_new_tokens):
            # Criar máscara causal para os tokens atuais
            seq_len = input_ids.shape[1]
            mask = self.generate_causal_mask(seq_len).to(input_ids.device)

            # Forward pass
            logits = self(input_ids, mask)

            # Pegar logits do último token
            next_token_logits = logits[:, -1, :] / temperature

            # Top-k sampling (se especificado)
            if top_k is not None:
                indices_to_remove = next_token_logits < torch.topk(next_token_logits, top_k)[0][..., -1, None]
                next_token_logits[indices_to_remove] = float('-inf')

            # Sample próximo token
            probs = F.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)

            # Adicionar à sequência
            input_ids = torch.cat([input_ids, next_token], dim=1)

        return input_ids



    """Modelo Transformer completo para geração de texto"""
    def __init__(self, vocab_size, d_model, num_heads, d_ff, num_layers,
                 max_seq_len=512, dropout=0.1):
        super().__init__()

        self.vocab_size = vocab_size
        self.d_model = d_model

        # 1. Embedding de tokens
        self.token_embedding = nn.Embedding(vocab_size, d_model)

        # 2. Positional Encoding (ISSO ESTAVA FALTANDO!)
        self.pos_encoding = PositionalEncoding(d_model, max_seq_len, dropout)

        # 3. Blocos Transformer
        self.transformer_blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

        # 4. Output Head
        self.output_head = nn.Linear(d_model, vocab_size)

        self.dropout = nn.Dropout(dropout)

        # Inicialização dos pesos (importante!)
        self._init_weights()

    def _init_weights(self):
        """Inicialização Xavier/Glorot para melhor convergência"""
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, x, mask=None):
        """
        x: tokens (batch, seq_len)
        mask: máscara causal para geração (batch, seq_len, seq_len)
        """
        # 1. Token embedding + scaling
        x = self.token_embedding(x) * math.sqrt(self.d_model)

        # 2. Adicionar positional encoding
        x = self.pos_encoding(x)

        # 3. Passar pelos blocos Transformer
        for block in self.transformer_blocks:
            x = block(x, mask)

        # 4. Projetar para vocabulário
        logits = self.output_head(x)

        return logits

    def generate_causal_mask(self, size):
        """Criar máscara causal (triangular inferior) para geração"""
        mask = torch.triu(torch.ones(size, size), diagonal=1).bool()
        mask = ~mask  # Invert for 1 = can see, 0 = cannot see
        return mask.unsqueeze(0).unsqueeze(0)  # (1, 1, size, size)

    @torch.no_grad()
    def generate(self, input_ids, max_new_tokens=50, temperature=1.0, top_k=None):
        """
        Gerar texto token por token
        input_ids: (batch, seq_len) - tokens iniciais
        """
        self.eval()
        batch_size = input_ids.shape[0]

        for _ in range(max_new_tokens):
            # Criar máscara causal para os tokens atuais
            seq_len = input_ids.shape[1]
            mask = self.generate_causal_mask(seq_len).to(input_ids.device)

            # Forward pass
            logits = self(input_ids, mask)

            # Pegar logits do último token
            next_token_logits = logits[:, -1, :] / temperature

            # Top-k sampling (se especificado)
            if top_k is not None:
                indices_to_remove = next_token_logits < torch.topk(next_token_logits, top_k)[0][..., -1, None]
                next_token_logits[indices_to_remove] = float('-inf')

            # Sample próximo token
            probs = F.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)

            # Adicionar à sequência
            input_ids = torch.cat([input_ids, next_token], dim=1)

        return input_ids

print("Configurações iniciais feitam!")

from collections import Counter, defaultdict
import re
import torch
from torch.utils.data import Dataset


class BPETokenizer:
    """
    Byte Pair Encoding tokenizer treinado diretamente no texto fornecido.
    Compatível com o TextDataset abaixo.
    """

    SPECIAL_TOKENS = ["<PAD>", "<UNK>", "<BOS>", "<EOS>"]

    def __init__(self, vocab_size: int = 512, min_freq: int = 2):
        """
        vocab_size : tamanho máximo do vocabulário (especiais + chars + merges)
        min_freq   : par precisa aparecer >= min_freq para ser fundido
        """
        self.target_vocab_size = vocab_size
        self.min_freq = min_freq

        self.word_to_idx: dict[str, int] = {}
        self.idx_to_word: dict[int, str] = {}
        self.merges: list[tuple[str, str]] = []   # ordem dos merges aprendidos
        self.vocab_size = 0

    # ------------------------------------------------------------------ #
    #  Treinamento                                                         #
    # ------------------------------------------------------------------ #

    def train(self, text: str) -> None:
        """Aprende o vocabulário BPE a partir de `text`."""
        # 1. Pré-tokeniza em palavras e representa cada uma como lista de chars
        #    O marcador "Ġ" (U+0120) indica início de palavra — estilo GPT-2
        words_raw = text.lower().split()
        word_freq: Counter = Counter(words_raw)

        # Representação inicial: cada palavra = tupla de chars + </w>
        vocab: dict[tuple, int] = {
            tuple(self._word_to_chars(w)): freq
            for w, freq in word_freq.items()
        }

        # 2. Vocabulário base = chars únicos
        base_chars: set[str] = set()
        for token_seq in vocab:
            base_chars.update(token_seq)

        # 3. Inicializa word_to_idx com especiais + chars base
        self.word_to_idx = {tok: i for i, tok in enumerate(self.SPECIAL_TOKENS)}
        for ch in sorted(base_chars):
            if ch not in self.word_to_idx:
                self.word_to_idx[ch] = len(self.word_to_idx)

        # 4. Loop BPE
        self.merges = []
        while len(self.word_to_idx) < self.target_vocab_size:
            pairs = self._get_pair_freqs(vocab)
            if not pairs:
                break

            best_pair, best_freq = max(pairs.items(), key=lambda x: x[1])
            if best_freq < self.min_freq:
                break

            # Funde o par e adiciona ao vocabulário
            new_token = "".join(best_pair)
            self.merges.append(best_pair)
            self.word_to_idx[new_token] = len(self.word_to_idx)

            # Aplica o merge em todo o vocab
            vocab = self._apply_merge(vocab, best_pair, new_token)

        self.vocab_size = len(self.word_to_idx)
        self.idx_to_word = {v: k for k, v in self.word_to_idx.items()}

        print(f"BPE treinado | vocab: {self.vocab_size} tokens "
              f"| merges: {len(self.merges)}")

    # ------------------------------------------------------------------ #
    #  Encode / Decode                                                     #
    # ------------------------------------------------------------------ #

    def encode(self, text: str) -> list[int]:
        """Texto → lista de ids."""
        ids = []
        for word in text.lower().split():
            subwords = self._tokenize_word(word)
            for sw in subwords:
                ids.append(self.word_to_idx.get(sw, self.word_to_idx["<UNK>"]))
        return ids

    def decode(self, ids: list[int]) -> str:
        """Lista de ids → texto, reconstruindo palavras a partir de subwords e adicionando espaços corretamente."""
        if not ids:
            return ""

        decoded_subwords = [self.idx_to_word.get(i, "<UNK>") for i in ids]

        # Filter out special tokens that should not appear in the final text output
        filtered_subwords = [s for s in decoded_subwords if s not in ["<UNK>", "<PAD>", "<BOS>", "<EOS>"]]

        # Join all filtered subwords into a single string
        full_string = "".join(filtered_subwords)

        # Replace the </w> marker with a space to separate reconstructed words
        # Then strip any leading/trailing spaces
        result = full_string.replace("</w>", " ").strip()

        return result

    # ------------------------------------------------------------------ #
    #  Helpers internos                                                    #
    # ------------------------------------------------------------------ #

    @staticmethod
    def _word_to_chars(word: str) -> list[str]:
        """'hello' → ['h','e','l','l','o</w>']"""
        if not word:
            return []
        chars = list(word)
        chars[-1] = chars[-1] + "</w>"   # marca fim de palavra
        return chars

    @staticmethod
    def _get_pair_freqs(vocab: dict) -> Counter:
        pairs: Counter = Counter()
        for token_seq, freq in vocab.items():
            for a, b in zip(token_seq, token_seq[1:]):
                pairs[(a, b)] += freq
        return pairs

    @staticmethod
    def _apply_merge(vocab: dict, pair: tuple[str, str],
                     new_token: str) -> dict:
        new_vocab: dict = {}
        a, b = pair
        for token_seq, freq in vocab.items():
            new_seq: list[str] = []
            i = 0
            while i < len(token_seq):
                if (i < len(token_seq) - 1
                        and token_seq[i] == a
                        and token_seq[i + 1] == b):
                    new_seq.append(new_token)
                    i += 2
                else:
                    new_seq.append(token_seq[i])
                    i += 1
            new_vocab[tuple(new_seq)] = freq
        return new_vocab

    def _tokenize_word(self, word: str) -> list[str]:
        """Aplica os merges aprendidos a uma única palavra."""
        chars = self._word_to_chars(word)
        if not chars:
            return []
        # Aplica merges na ordem em que foram aprendidos
        for a, b in self.merges:
            new_chars: list[str] = []
            i = 0
            while i < len(chars):
                if i < len(chars) - 1 and chars[i] == a and chars[i + 1] == b:
                    new_chars.append(a + b)
                    i += 2
                else:
                    new_chars.append(chars[i])
                    i += 1
            chars = new_chars
        return chars


# ────────────────────────────────────────────────────────────────────────────
#  Dataset compatível com o restante do pipeline
# ────────────────────────────────────────────────────────────────────────────

class TextDataset(Dataset):
    def __init__(self, text: str, seq_len: int = 64,
                 bpe_vocab_size: int = 512, min_freq: int = 2):
        self.seq_len = seq_len

        # Treina o tokenizador BPE no texto fornecido
        self.tokenizer = BPETokenizer(vocab_size=bpe_vocab_size,
                                      min_freq=min_freq)
        self.tokenizer.train(text)

        # Expõe vocab_size para o modelo
        self.vocab_size = self.tokenizer.vocab_size

        # Tokeniza o texto completo
        self.tokens: list[int] = self.tokenizer.encode(text)
        print(f"Total de tokens no texto: {len(self.tokens)}")

    # Mantém a mesma interface de antes ──────────────────────────────────
    def encode(self, text: str) -> list[int]:
        return self.tokenizer.encode(text)

    def decode(self, ids: list[int]) -> str:
        return self.tokenizer.decode(ids)

    def __len__(self) -> int:
        return max(1, len(self.tokens) - self.seq_len)

    def __getitem__(self, idx: int):
        end_idx   = min(idx + self.seq_len, len(self.tokens) - 1)
        start_idx = end_idx - self.seq_len

        X = torch.tensor(self.tokens[start_idx:end_idx],     dtype=torch.long)
        y = torch.tensor(self.tokens[start_idx + 1:end_idx + 1], dtype=torch.long)

        if len(X) < self.seq_len:
            pad = self.seq_len - len(X)
            X = torch.cat([X, torch.zeros(pad, dtype=torch.long)])
            y = torch.cat([y, torch.zeros(pad, dtype=torch.long)])

        return X, y

print("Rodou com sucesso")

def generate(
    model,
    prompt,
    dataset,
    max_new_tokens=20,
    temperature=1.0,
    top_k=None,
    top_p=None,
    device='cpu'
):
    """
    Gera texto com opções avançadas de sampling

    Args:
        model: Modelo treinado
        prompt: Texto inicial
        dataset: Instância do TextDataset
        max_new_tokens: Número máximo de tokens a gerar
        temperature: Temperatura para sampling (mais alta = mais criativo)
        top_k: Mantém apenas os k tokens mais prováveis
        top_p: Nucleus sampling (mantém tokens com probabilidade acumulada p)
        device: Dispositivo
    """
    model.eval()

    # Tokenização do prompt inicial
    # Use tokenizer.encode directly to get the initial token IDs
    initial_ids = dataset.tokenizer.encode(prompt)

    # Adiciona token BOS se disponível e se ainda não for o primeiro token
    if '<BOS>' in dataset.tokenizer.word_to_idx and (not initial_ids or initial_ids[0] != dataset.tokenizer.word_to_idx['<BOS>']):
        initial_ids = [dataset.tokenizer.word_to_idx['<BOS>']] + initial_ids

    # Convert initial_ids to a tensor for the model
    input_ids = torch.tensor([initial_ids], dtype=torch.long).to(device)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            # Mantém apenas os últimos seq_len tokens para o forward pass
            # Isso é importante para que o modelo sempre veja uma sequência de tamanho fixo se seq_len for menor que o input_ids atual
            current_sequence_for_model = input_ids if input_ids.shape[1] <= dataset.seq_len else input_ids[:, -dataset.seq_len:]

            # Gera máscara causal para a sequência atual
            mask = model.generate_causal_mask(current_sequence_for_model.shape[1]).to(device)

            # Forward pass
            logits = model(current_sequence_for_model, mask)

            # Pega logits do último token e aplica temperatura
            next_token_logits = logits[:, -1, :] / temperature

            # Aplica top-k sampling
            if top_k is not None and top_k > 0:
                # Filtra os k tokens mais prováveis
                indices_to_remove = next_token_logits < torch.topk(next_token_logits, top_k)[0][..., -1, None]
                next_token_logits[indices_to_remove] = -float('Inf')

            # Aplica top-p (nucleus) sampling
            if top_p is not None and top_p < 1.0:
                sorted_logits, sorted_indices = torch.sort(next_token_logits, descending=True)
                cumulative_probs = torch.cumsum(torch.softmax(sorted_logits, dim=-1), dim=-1)

                # Remove tokens com probabilidade acumulada acima de top_p
                # Garante que pelo menos um token seja mantido
                sorted_indices_to_remove = cumulative_probs > top_p
                sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
                sorted_indices_to_remove[..., 0] = 0 # Sempre mantém o token mais provável

                indices_to_remove = sorted_indices[sorted_indices_to_remove]
                next_token_logits[indices_to_remove] = -float('Inf')

            # Calcula probabilidades após sampling
            probs = torch.softmax(next_token_logits, dim=-1)

            # Amostra o próximo token
            next_token_id = torch.multinomial(probs, num_samples=1) # Tensor de shape (batch_size, 1)

            # Converte o ID do token para string para verificar EOS/PAD
            next_word_str = dataset.tokenizer.idx_to_word.get(next_token_id.item(), '<UNK>') # Usar .get para segurança

            # Para a geração se EOS ou PAD for gerado
            if next_word_str in ['<EOS>', '<PAD>']:
                break

            # Adiciona o ID do token à sequência total gerada
            input_ids = torch.cat([input_ids, next_token_id], dim=1)

    # Decodifica a sequência completa de IDs gerada em uma string limpa
    final_generated_ids = input_ids[0].tolist()

    # Remove o token BOS do início se ele foi adicionado para a geração e não é desejado na saída final
    if '<BOS>' in dataset.tokenizer.word_to_idx and final_generated_ids and final_generated_ids[0] == dataset.tokenizer.word_to_idx['<BOS>']:
      final_generated_ids = final_generated_ids[1:]

    return dataset.tokenizer.decode(final_generated_ids)


def test_generation(model, dataset, device='cpu'):
    """Testa geração com diferentes prompts"""

    test_prompts = [
        "era uma vez",
        "o programador",
        "a floresta",
        "para treinar"
    ]

    print("=" * 50)
    print("TESTE DE GERAÇÃO")
    print("=" * 50)

    for prompt in test_prompts:
        print(f"\nPrompt: '{prompt}'")

        # Geração com temperatura baixa (mais determinística)
        generated = generate(
            model,
            prompt,
            dataset,
            max_new_tokens=10,
            temperature=0.7,
            device=device
        )
        print(f"T=0.7: {generated}")

        # Geração com temperatura alta (mais criativa)
        generated = generate(
            model,
            prompt,
            dataset,
            max_new_tokens=10,
            temperature=1.2,
            device=device
        )
        print(f"T=1.2: {generated}")

        print("-" * 30)

print("Rodou com sucesso!")

# ── Dataset ──────────────────────────────────────────────────────────────
text_ = """
O reino de Eldoria estava em silêncio naquela noite. As tochas iluminavam as muralhas enquanto os guardas observavam a floresta escura além dos portões.

Luna caminhava lentamente pela biblioteca antiga. Entre livros esquecidos e mapas rasgados, ela procurava o artefato perdido dos magos.

Ao abrir um velho pergaminho, símbolos dourados começaram a brilhar. O chão tremeu levemente e uma voz ecoou pelos corredores:

“Somente aquele que domina o tempo poderá despertar o coração da montanha.”

Enquanto isso, no norte gelado, o exército de Ravok avançava pelas planícies cobertas de neve. Os soldados marchavam em silêncio absoluto.

Kael observava o céu vermelho no horizonte. Algo estranho estava acontecendo. Os pássaros haviam desaparecido e o vento parecia carregar sussurros antigos.

Na aldeia distante, crianças corriam entre as casas de madeira enquanto os ferreiros trabalhavam sem descanso. Todos temiam a chegada do inverno eterno.
"""

text = """
def soma(a, b):
    return a + b

def multiplicar(x, y):
    resultado = x * y
    return resultado

def divisao(a, b):
    if b == 0:
        return "Erro: divisao por zero"
    return a / b

def subtracao(a, b):
    return a - b

def potencia(base, expoente):
    resultado = 1
    for _ in range(expoente):
        resultado *= base
    return resultado

def fibonacci(n):
    if n <= 0:
        return []
    elif n == 1:
        return [0]
    elif n == 2:
        return [0, 1]
    seq = [0, 1]
    for i in range(2, n):
        seq.append(seq[-1] + seq[-2])
    return seq

def eh_par(numero):
    return numero % 2 == 0

def fatorial(n):
    if n < 0:
        return None
    resultado = 1
    for i in range(1, n + 1):
        resultado *= i
    return resultado

class Pessoa:
    def __init__(self, nome):
        self.nome = nome

    def apresentar(self):
        return f"Olá, meu nome é {self.nome}"

    def mudar_nome(self, novo_nome):
        self.nome = novo_nome

class Carro:
    def __init__(self, marca, modelo, ano):
        self.marca = marca
        self.modelo = modelo
        self.ano = ano
        self.velocidade = 0

    def acelerar(self, incremento):
        self.velocidade += incremento
        return f"Velocidade atual: {self.velocidade} km/h"

    def frear(self, decremento):
        self.velocidade = max(0, self.velocidade - decremento)
        return f"Velocidade atual: {self.velocidade} km/h"

    def descricao(self):
        return f"{self.marca} {self.modelo} ({self.ano})"

class ContaBancaria:
    def __init__(self, titular, saldo_inicial=0):
        self.titular = titular
        self.saldo = saldo_inicial
        self.historico = []

    def depositar(self, valor):
        if valor > 0:
            self.saldo += valor
            self.historico.append(f"Depósito: +{valor}")
            return f"Depósito de R${valor} realizado. Saldo: R${self.saldo}"
        return "Valor inválido"

    def sacar(self, valor):
        if 0 < valor <= self.saldo:
            self.saldo -= valor
            self.historico.append(f"Saque: -{valor}")
            return f"Saque de R${valor} realizado. Saldo: R${self.saldo}"
        return "Saldo insuficiente ou valor inválido"

    def extrato(self):
        return f"Titular: {self.titular}\\nSaldo: R${self.saldo}\\nHistórico: {self.historico}"
"""

print("Rodou com sucesso!")


from torch.utils.data import random_split

if __name__ == "__main__":

    device  = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using: {device}")



    # ── Bônus: ajusta hiperparâmetros ao tamanho do dataset ──────────────────
    dataset = TextDataset(text, seq_len=64)   # sua classe de dataset
    n_samples = len(dataset)
    vocab_size = dataset.vocab_size

    # Regras simples: dataset pequeno → modelo menor; grande → modelo maior
    if n_samples < 500:
        d_model, num_heads, d_ff, num_layers = 64,  2,  256, 2
    elif n_samples < 5_000:
        d_model, num_heads, d_ff, num_layers = 128, 4,  512, 4
    else:
        d_model, num_heads, d_ff, num_layers = 256, 8, 1024, 6

    print(f"Dataset: {n_samples} amostras → d_model={d_model}, layers={num_layers}")



    # ── Split treino / validação ──────────────────────────────────────────────
    train_size = int(0.8 * n_samples)
    val_size   = n_samples - train_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])



    # ── Configurações para infraestrutura pequena ─────────────────────────────
    micro_batch_size    = 4          # cabe na GPU/CPU pequena
    accumulation_steps  = 8          # batch efetivo = 4 × 8 = 32
    effective_batch_size = micro_batch_size * accumulation_steps

    train_loader = DataLoader(train_dataset, batch_size=micro_batch_size, shuffle=True)
    val_loader   = DataLoader(val_dataset,   batch_size=micro_batch_size, shuffle=False)




    # ── Modelo ────────────────────────────────────────────────────────────────
    model = TransformerModel(
        vocab_size=vocab_size,
        d_model=d_model,
        num_heads=num_heads,
        d_ff=d_ff,
        num_layers=num_layers,
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

    epochs = 1000

    # Alternativa: scheduler baseado em validação (recomendado para datasets pequenos)
    # Calcular corretamente o número de steps de otimização por época
    steps_per_optimizer_step = len(train_loader) // accumulation_steps

    # Se for 0, ajustamos o accumulation_steps ou usamos 1 como mínimo
    if steps_per_optimizer_step == 0:
        print(f"AVISO: accumulation_steps ({accumulation_steps}) > batches ({len(train_loader)})")
        print(f"Ajustando accumulation_steps para {len(train_loader)}")
        accumulation_steps = len(train_loader)
        steps_per_optimizer_step = 1

    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=5e-4,
        steps_per_epoch=steps_per_optimizer_step,  # agora será pelo menos 1
        epochs=epochs,
        pct_start=0.1,
    )



    # ── Early Stopping (sem salvar checkpoint) ────────────────────────────────
    best_val_loss    = float("inf")
    patience         = 10
    patience_counter = 0



    # ── Loop de treino ────────────────────────────────────────────────────────
    for epoch in range(epochs):

        # — Treino com gradient accumulation —
        model.train()
        optimizer.zero_grad()

        for step, batch in enumerate(train_loader):
            X, y = batch
            X, y = X.to(device), y.to(device)

            mask   = model.generate_causal_mask(X.shape[1]).to(device)
            logits = model(X, mask)
            loss   = criterion(logits.view(-1, vocab_size), y.view(-1))

            # Normaliza pelo nº de passos acumulados
            (loss / accumulation_steps).backward()

            if (step + 1) % accumulation_steps == 0:
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

        # — Validação —
        model.eval()
        val_loss_total = 0.0

        with torch.no_grad():
            for batch in val_loader:
                X, y = batch
                X, y = X.to(device), y.to(device)

                mask   = model.generate_causal_mask(X.shape[1]).to(device)
                logits = model(X, mask)
                val_loss_total += criterion(logits.view(-1, vocab_size), y.view(-1)).item()

        val_loss = val_loss_total / len(val_loader)

        if epoch % 100 == 0:
            print(f"Epoch {epoch+1}/{epochs} | train_loss={loss.item():.4f} | val_loss={val_loss:.4f}")

        # — Early Stopping —
        if val_loss < best_val_loss:
            best_val_loss    = val_loss
            patience_counter = 0          # melhora → reseta contador
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping na época {epoch+1} (val_loss não melhorou por {patience} épocas)")
                break



import re
import time
import sys

ESC = "\033["

class C:
    RESET      = f"{ESC}0m"
    BOLD       = f"{ESC}1m"
    KEYWORD    = f"{ESC}38;5;204m"
    STRING     = f"{ESC}38;5;114m"
    COMMENT    = f"{ESC}38;5;244m"
    NUMBER     = f"{ESC}38;5;221m"
    DECORATOR  = f"{ESC}38;5;215m"
    BUILTIN    = f"{ESC}38;5;81m"
    CLASS_NAME = f"{ESC}38;5;229m"
    FUNC_NAME  = f"{ESC}38;5;153m"
    SELF       = f"{ESC}38;5;210m"
    DEFAULT    = f"{ESC}38;5;255m"
    BG         = f"{ESC}48;5;235m"
    HEADER     = f"{ESC}38;5;39m"


KEYWORDS = {
    "False", "None", "True", "and", "as", "assert", "async", "await",
    "break", "class", "continue", "def", "del", "elif", "else", "except",
    "finally", "for", "from", "global", "if", "import", "in", "is",
    "lambda", "nonlocal", "not", "or", "pass", "raise", "return", "try",
    "while", "with", "yield"
}

BUILTINS = {
    "print", "len", "range", "int", "str", "float", "list", "dict",
    "tuple", "set", "bool", "type", "isinstance", "hasattr", "getattr",
    "setattr", "super", "property", "staticmethod", "classmethod",
    "enumerate", "zip", "map", "filter", "sorted", "reversed", "sum",
    "min", "max", "abs", "round", "open", "input", "format", "repr",
    "append", "extend", "insert", "remove", "pop", "update"
}


def colorize_line(line: str) -> str:
    if re.match(r'^\s*#', line):
        return C.COMMENT + line + C.RESET

    if re.match(r'^\s*@\w+', line):
        return C.DECORATOR + line + C.RESET

    result = []
    i = 0
    n = len(line)

    while i < n:
        # Strings (f-string ou normal)
        str_match = re.match(
            r'(f?)("""[\s\S]*?"""|\'\'\'[\s\S]*?\'\'\'|"[^"\n]*"|\'[^\'\n]*\')',
            line[i:]
        )
        if str_match:
            result.append(C.STRING + str_match.group(0) + C.DEFAULT)
            i += str_match.end()
            continue

        # Número
        num_match = re.match(r'\d+(\.\d+)?', line[i:])
        if num_match and (i == 0 or not line[i-1].isalnum() and line[i-1] != '_'):
            result.append(C.NUMBER + num_match.group(0) + C.DEFAULT)
            i += num_match.end()
            continue

        # Palavra / identificador
        word_match = re.match(r'[A-Za-z_]\w*', line[i:])
        if word_match:
            word = word_match.group(0)
            if word in KEYWORDS:
                result.append(C.KEYWORD + word + C.DEFAULT)
            elif word == "self":
                result.append(C.SELF + word + C.DEFAULT)
            elif word in BUILTINS:
                result.append(C.BUILTIN + word + C.DEFAULT)
            else:
                prev_plain = re.sub(r'\033\[[^m]*m', '', "".join(result)).rstrip()
                if prev_plain.endswith("class"):
                    result.append(C.CLASS_NAME + word + C.DEFAULT)
                elif prev_plain.endswith("def"):
                    result.append(C.FUNC_NAME + word + C.DEFAULT)
                else:
                    result.append(C.DEFAULT + word)
            i += word_match.end()
            continue

        # Qualquer outro char
        result.append(C.DEFAULT + line[i])
        i += 1

    return "".join(result) + C.RESET


def format_raw_code(raw: str) -> str:
    code = re.sub(r'[ \t]+', ' ', raw.strip())

    block_kws = r'\b(class |def |if |elif |else:|for |while |try:|except[\s:]|finally:|with |return |raise |pass\b|break\b|continue\b|import |from )'
    code = re.sub(block_kws, r'\n\1', code)

    raw_lines = [l.strip() for l in code.split('\n') if l.strip()]

    indented = []
    indent = 0
    for line in raw_lines:
        if re.match(r'^(else:|elif |except[\s:]|finally:)', line):
            indent = max(0, indent - 1)

        indented.append('    ' * indent + line)

        if line.endswith(':') and not line.startswith('#'):
            indent += 1

        if re.match(r'^(return|pass|break|continue|raise)\b', line):
            indent = max(0, indent - 1)

    return '\n'.join(indented)


def stream_code(
    raw_output: str,
    char_delay: float = 0.010,
    line_delay: float = 0.06,
    lang: str = "python",
    title: str = "Código gerado",
):
    width = 62
    bar   = C.HEADER + C.BOLD + '─' * width + C.RESET
    label = C.HEADER + C.BOLD + f'  {lang.upper()}  ›  {title}' + C.RESET

    print()
    print(bar)
    print(label)
    print(bar)
    print()

    formatted = format_raw_code(raw_output)
    lines = formatted.split('\n')

    for line in lines:
        colored = C.BG + colorize_line(line)

        # Separa sequências ANSI dos chars visíveis → delay só no texto
        tokens = re.split(r'(\033\[[^m]*m)', colored)

        for tok in tokens:
            if tok.startswith('\033['):
                sys.stdout.write(tok)
                sys.stdout.flush()
            else:
                for ch in tok:
                    sys.stdout.write(ch)
                    sys.stdout.flush()
                    time.sleep(char_delay)

        sys.stdout.write(C.RESET + '\n')
        sys.stdout.flush()
        time.sleep(line_delay)

    print()
    print(bar)
    print(C.COMMENT + f'  ✓ {len(lines)} linhas  |  {lang}' + C.RESET)
    print(bar)
    print()


# ─────────────────────────────────────────
# Exemplo
# ─────────────────────────────────────────
# if __name__ == '__main__':
#     raw = (
#         "class ContaBancaria: def __init__(self, titular, saldo_inicial=0): "
#         "self.titular = titular self.saldo = saldo_inicial self.historico = [] "
#         "def depositar(self, valor): if valor > 0: self.saldo += valor "
#         "self.historico.append(f'deposito: +{valor}') "
#         "return f'deposito de R$ realizado. Saldo: {self.saldo}' "
#         "return 'Valor invalido' "
#         "def sacar(self, valor): if 0 < valor <= self.saldo: self.saldo -= valor "
#         "self.historico.append(f'saque: -{valor}') "
#         "return f'saque realizado. Saldo: {self.saldo}' "
#         "return 'Saldo insuficiente' "
#         "def ver_saldo(self): return f'Titular: {self.titular} | Saldo: {self.saldo}'"
#     )

#     stream_code(raw, char_delay=0.008, line_delay=0.05, title="ContaBancaria")


                # Após o treinamento, gere texto:

tmp = 1.0
max_token = 120
prompt = """
class Carro:
    def __init__(self, marca, modelo, ano):
        self.marca = marca
        self.modelo = modelo
        self.ano = ano
        self.velocidade = 0
"""

output = generate(
    model=model,
    prompt=prompt,
    dataset=dataset,  # Passe o dataset que contém os mappings
    max_new_tokens=max_token,
    temperature=tmp,
    device=device
)


print(
    f" temperatura: {tmp}\n",
    f"max token: {max_token}\n",
    f"prompt: {prompt}\n",
    "---"*10,
)

stream_code(
    raw_output=output,
    char_delay=0.001,   # mais rápido/lento por char
    line_delay=0.001,    # pausa entre linhas
    title="Meu Código"
)

# Ou use a função de teste:
#test_generation(model, dataset, device)

Configurações iniciais feitam!
Rodou com sucesso
Rodou com sucesso!
Rodou com sucesso!
Using: cpu
BPE treinado | vocab: 295 tokens | merges: 211
Total de tokens no texto: 462
Dataset: 398 amostras → d_model=64, layers=2
Epoch 1/1000 | train_loss=6.0191 | val_loss=5.9515
Epoch 101/1000 | train_loss=0.0932 | val_loss=0.0904
Early stopping na época 122 (val_loss não melhorou por 10 épocas)
 temperatura: 1.0
 max token: 120
 prompt: 
class Carro:
    def __init__(self, marca, modelo, ano):
        self.marca = marca
        self.modelo = modelo
        self.ano = ano
        self.velocidade = 0

 ------------------------------

──────────────────────────────────────────────────────────────
  PYTHON  ›  Meu Código
──────────────────────────────────────────────────────────────

class carro:
    def __init__(self, marca, modelo, ano): self.marca = marca self.modelo = modelo self.ano = ano self.velocidade = 0
    def acelerar(self, incremento): self.velocidade += incremento
    return f"veloci

In [23]:
tmp = 1.0
max_token = 110
prompt = """
class Carro:
    def __init__(self, marca, modelo, ano):
        self.marca = marca
        self.modelo = modelo
        self.ano = ano
        self.velocidade = 0
"""

output = generate(
    model=model,
    prompt=prompt,
    dataset=dataset,  # Passe o dataset que contém os mappings
    max_new_tokens=max_token,
    temperature=tmp,
    device=device
)


print(
    f" temperatura: {tmp}\n",
    f"max token: {max_token}\n",
    f"prompt: {prompt}\n",
    "---"*10,
)

stream_code(
    raw_output=output,
    char_delay=0.007,   # mais rápido/lento por char
    line_delay=0.004,    # pausa entre linhas
    title="Meu Código"
)

 temperatura: 1.0
 max token: 110
 prompt: 
class Carro:
    def __init__(self, marca, modelo, ano):
        self.marca = marca
        self.modelo = modelo
        self.ano = ano
        self.velocidade = 0

 ------------------------------

──────────────────────────────────────────────────────────────
  PYTHON  ›  Meu Código
──────────────────────────────────────────────────────────────

class carro:
    def __init__(self, marca, modelo, ano): self.marca = marca self.modelo = modelo self.ano = ano self.velocidade = 0
    def acelerar(self, incremento): self.velocidade += incremento
    return f"velocidade atual: {self.velocidade} km/h"
def frear(self, decremento): self.velocidade = max(0, self.velocidade - decremento)
return f"velocidade atual: {self.velocidade} km/h"
def descricao(self):
    return f"{self.marca} {self.modelo} ({self.ano})"
class contabancaria:
    def __init__(self, titular, saldo_inicial=0): self.titular = titular self.saldo = saldo_inicial self.historico = []
   

In [21]:
# Salvar o modelo
torch.save(model, "modelo_completo.pth")


In [22]:
# Carega o modelo:

model = torch.load("modelo_completo.pth")
model.eval()

UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL __main__.TransformerModel was not an allowed global by default. Please use `torch.serialization.add_safe_globals([__main__.TransformerModel])` or the `torch.serialization.safe_globals([__main__.TransformerModel])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn more about t